# モンテカルロ・ステップ：コードの深掘り解説

このノートでは、シミュレーションの心臓部である `mcs_step` メソッドのコードを、一行ずつ徹底的に解説します。

---

## 1. 全体の構造：なぜ `L * L` 回ループするのか？

```python
def mcs_step(self):
    for _ in range(self.N):  # self.N = L * L
```

- **コードの意味**: 格子の全サイト数（面積）と同じ回数だけ、以下の「更新試行」を繰り返します。
- **物理的な意味**: これにより、平均して全ての場所のスピンが1回ずつ「ひっくり返るチャンス」を得ます。これが「1ステップ」という時間の単位になります。

---

## 2. サイトの選択：どこを選ぶ？

```python
i = np.random.randint(0, self.L)  # 行(y)をランダムに選ぶ
j = np.random.randint(0, self.L)  # 列(x)をランダムに選ぶ
```

- **なぜランダム？**: 順番に選ぶ（スキャンする）方法もありますが、ランダムに選ぶ方が物理的な「熱によるゆらぎ」を正しくシミュレートでき、統計的な偏りが出にくいとされています。

---

## 3. 隣接スピンの計算：周期境界条件のトリック

```python
neighbor_sum = (
    self.spins[(i+1)%self.L, j] +  # 下
    self.spins[(i-1)%self.L, j] +  # 上
    self.spins[i, (j+1)%self.L] +  # 右
    self.spins[i, (j-1)%self.L]    # 左
)
```

- **`% self.L` (余り) の役割**: 
    - もし `i` が右端（`L-1`）のとき、`i+1` は `L` になります。
    - `L % L` は `0` になるので、自動的に左端の `0` 番目に戻ります。
- **物理的な意味**: これにより、格子に「端」がなくなり、ドーナツの表面のように無限に続く平面をシミュレートできます（境界の影響を消すため）。

---

## 4. エネルギー変化 $\Delta E$ の計算

```python
dE = 2 * self.J * S * neighbor_sum
```

- **なぜ `2` がつくの？**:
    - 今のエネルギーは $-J \cdot S \cdot (周囲の和)$ です。
    - $S$ を $-S$ にひっくり返すと、エネルギーは $+J \cdot S \cdot (周囲の和)$ になります。
    - その「差（あとーさき）」を計算すると、 $J - (-J) = 2J$ となるため、`2` 倍の差が生まれます。

---

## 5. メトロポリス判定：運命の分かれ道

```python
if dE <= 0 or np.random.rand() < np.exp(-dE / self.T):
    self.spins[i, j] *= -1
```

ここが一番面白いところです。

1. **`dE <= 0`**: エネルギーが下がる（安定する）なら、**無条件で採用**。磁石として揃おうとする力です。
2. **`np.random.rand() < np.exp(-dE / self.T)`**: 
    - エネルギーが上がる（不安定になる）場合でも、サイコロを振って合格すれば採用します。
    - **温度 $T$ が高い** $\to$ `exp` の値が大きくなる $\to$ 合格しやすくなる $\to$ **バラバラになる**。
    - **温度 $T$ が低い** $\to$ `exp` の値が小さくなる $\to$ 合格しにくくなる $\to$ **揃ったままになる**。

--- 

### まとめ
この短いコードの中に、「磁石が揃おうとする性質」と「熱がかき乱そうとする性質」のせめぎ合いが全て凝縮されています。